# 使用@tool装饰器定义工具

## 1、 自定义工具描述：description

举例1：

函数使用@tool装饰器修饰以后，就是一个可以被模型识别的工具

如下的程序报错，因为：
在没有提供description参数的情况下，要求函数必须提供docstring。

In [ ]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city : str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

修改为

In [4]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city : str):
    """
    查询城市天气
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市天气\n\nArgs:\n    city : 具体的城市',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：使用description参数

In [3]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city : str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例3：

在同时声明了description和docstring的情况下，description的优先级更高

In [ ]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city : str):
    """获取城市天气"""
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

举例4：

当我们没有向 @tool 传递 description 参数时，默认情况下，tool 会将 docstring 整体视为description

In [6]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city : str):
    """
    获取城市天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

举例5：

In [9]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True,description="获取具体城市的天气")
def get_weather(city : str):
    """
    获取具体城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2、更改工具名称： name_or_callable

举例：

开发中，习惯使用函数名作为工具名称，不推荐自定义工具名称。

In [ ]:
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True,name_or_callable="getWeather")
def get_weather(city : str):
    """
    获取具体城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

## 3、自定义args_schema

### 3.1 使用Pydantic模型定义

举例1：

In [14]:
from pydantic import BaseModel
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


class WeatherInput(BaseModel):
    city : str

@tool(args_schema=WeatherInput)
def get_weather(city : str):
    """
    获取具体城市的天气
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {'properties': {'city': {'type': 'integer'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：

In [15]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


class WeatherInput(BaseModel):
    city : str = Field(description="城市",default="北京.")

@tool(args_schema=WeatherInput)
def get_weather(city : str):
    """
    获取具体城市的天气
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

举例3：

In [17]:
from typing import Literal


class WeatherInput(BaseModel):
    city : str = Field(description="城市",default="北京")
    unit : Literal["celsius", "fahrenheit"]
    include_forecast : bool = Field(
        description="是否包含未来五天的天气预报",
        default=False
    )


@tool(args_schema=WeatherInput)
def get_weather(city : str,unit : str = "celsius",include_forecast : bool = False):
    """
    获取具体城市的天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

### 3.2 使用Json schema定义

In [4]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

json_schema = {
    "properties": {
        "city": {"default": "北京", "description": "城市", "type": "string"},
        "unit": {"description": "温度单位", "enum": ["celsius", "fahrenheit"], "type": "string"},
        "include_forecast": {
            "default": False,
            "description": "是否包含未来五天的天气预报",
            "type": "boolean",
        },
    },
    "required": ["unit"],
    "type": "object",
}


@tool(args_schema=json_schema)
def get_weather(city: str = "北京", unit: str = "celsius", include_forecast: bool = False):
    """
    获取具体城市的天气
    """
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市', 'type': 'string'},
                'unit': {'description': '温度单位', 'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}